<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_more_about_joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Advanced: More about Joins


## **SQL Environment Setup (do not edit)**

In [14]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [15]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
DROP TABLE IF EXISTS dictionary;

CREATE TABLE dictionary (
    entry_id INTEGER PRIMARY KEY,
    word VARCHAR NOT NULL,
    hypernym_id INTEGER,
   -- FOREIGN KEY (hypernym_id) REFERENCES dictionary(entry_id)
);

INSERT INTO dictionary (entry_id, word, hypernym_id) VALUES
(3, 'color', NULL),
(7, 'gemstone', NULL),
(16, 'animal', NULL),

(2, 'carnivore', 16),
(5, 'blue', 3),
(11, 'bird', 16),
(17, 'fish', 16),

(1, 'bear', 2),
(4, 'red', 3),
(6, 'yellow', 3),
(8, 'diamond', 7),
(9, 'ruby', 7),
(10, 'emerald', 7),
(12, 'pelican', 11),
(13, 'sparrow', 11),
(14, 'cyan', 5),
(15, 'marine', 5),
(18, 'salmon', 17),
(19, 'trout', 18);

DROP TABLE IF EXISTS buyer;
DROP TABLE IF EXISTS item;

CREATE TABLE item (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    price INTEGER NOT NULL,
    type VARCHAR NOT NULL
);

CREATE TABLE buyer (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    funds INTEGER NOT NULL
);

INSERT INTO item (id, name, price, type) VALUES
(1, 'Upon the Sky', 6000, 'book'),
(2, 'Starless', 15000, 'painting'),
(3, 'Glimmering Hope', 11000, 'painting'),
(4, 'Seconds of Hope', 3000, 'sculpture'),
(5, 'Unexpexted Gift', 9000, 'sculpture'),
(6, 'The Thinker', 7000, 'sculpture'),
(7, 'White Soul', 85000, 'book'),
(8, 'Forgiveness', 12500, 'book'),
(9, 'Acceptance', 21000, 'painting'),
(10, 'Blue Teapot', 50000, 'dinnerware');

INSERT INTO buyer (id, name, funds) VALUES
(1, 'Mark Gatis', 15000),
(2, 'Michal Bar', 21000),
(3, 'Harvey Trust', 5000),
(4, 'John Martin', 7500),
(5, 'Marin Gant', 95000);

DROP TABLE IF EXISTS couples;
DROP TABLE IF EXISTS apartments;

CREATE TABLE apartments (
    id INTEGER PRIMARY KEY,
    location VARCHAR NOT NULL,
    price INTEGER NOT NULL,
    rating INTEGER NOT NULL
);

CREATE TABLE couples (
    id INTEGER PRIMARY KEY,
    couple_name VARCHAR NOT NULL,
    min_price INTEGER NOT NULL,
    max_price INTEGER NOT NULL,
    pref_location VARCHAR NOT NULL
);

INSERT INTO apartments (id, location, price, rating) VALUES
(1, 'Chicago', 300000, 3),
(2, 'Charleston', 400000, 4),
(3, 'Chicago', 200000, 3),
(4, 'Las Vegas', 250000, 3),
(5, 'Las Vegas', 150000, 4),
(6, 'Seattle', 100000, 4),
(7, 'Seattle', 140000, 3),
(8, 'San Francisco', 260000, 3),
(9, 'San Francisco', 280000, 4),
(10, 'San Francisco', 350000, 5);

INSERT INTO couples (id, couple_name, min_price, max_price, pref_location) VALUES
(1, 'Mark and Jess', 120000, 180000, 'Charleston'),
(2, 'Jenny and Pawel', 130000, 220000, 'Chicago'),
(3, 'Peter and Jenny', 210000, 500000, 'Las Vegas'),
(4, 'Jessica and Pawel', 220000, 280000, 'Las Vegas'),
(5, 'John and Maria', 110000, 190000, 'Seattle'),
(6, 'Maria and Sam', 150000, 210000, 'Seattle'),
(7, 'Samantha and Theo', 220000, 250000, 'San Francisco'),
(8, 'Mark and Jenny', 110000, 120000, 'San Francisco'),
(9, 'August and Marta', 150000, 170000, 'Chicago'),
(10, 'Isabelle and Mathew', 270000, 360000, 'Chicago'),
(11, 'Jacob and Alice', 280000, 350000, 'San Francisco'),
(12, 'Alice and John', 290000, 640000, 'San Francisco');
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


## Self-joins exercises
In the next couple of exercises, we'll work with words in a dictionary. Do you know that linguists classify words?

Our table contains some English words and their hypernyms.

A hypernym is a term with a broad meaning, that "contains" other terms. For example, the hypernym of "sparrow" is "bird".  


In [16]:
# @title Get to know the Dictionary table
base_gtk_dictionary = make_df_validator_nospoilers(
    expected_hash='c53fb2f98ec53273d2937348c1fdd90f4da78c9afc18f38c2d292e9b7614484e',
    required_cols=['entry_id', 'word', 'hypernym_id'],
    expected_rows=19,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_gtk_dictionary = base_gtk_dictionary

make_sql_runner(
    conn,
    runner_id="gtk_dictionary",
    description_md='### Dictionary table\nSelect all data from the dictionary table. It contains the following columns:\n\n  * entry_id - the ID of an entry in the dictionary,\n  * word - the word associated with a particular entry,\n  * hypernym_id - the ID of an entry that is a hypernym for a particular word.\n',
    validator=val_gtk_dictionary,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['dictionary']
)


In [17]:
# @title Practice 1
base_practice_1 = make_df_validator_nospoilers(
    expected_hash='4db3e4ca28dd87ee40b3514b49b396ee7a78d95d5984f5824cb7e9ff32d3b98b',
    required_cols=['entry', 'hypernym'],
    expected_rows=16,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_1 = base_practice_1

make_sql_runner(
    conn,
    runner_id="practice_1",
    description_md='### Practice 1\nWrite a query that lists each word together with the word representing its **direct hypernym**.\n\n* The result must contain two columns:\n\n  * entry — the value of word for the dictionary entry.\n  * hypernym — the word of the row referenced by that entry’s hypernym_id.\n\n Only include entries that **have a hypernym** (i.e., hypernym_id is not NULL).\n',
    validator=val_practice_1,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['dictionary']
)


In [18]:
# @title Practice 2
base_practice_2 = make_df_validator_nospoilers(
    expected_hash='0cd614c32c0e50bb7ff3609acf2271ead2d815b30cd57daa119bbb4a73e9ba6d',
    required_cols=['entry', 'hypernym', 'grandhypernym'],
    expected_rows=7,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_2 = base_practice_2

make_sql_runner(
    conn,
    runner_id="practice_2",
    description_md='### Practice 2\nWrite a query that lists each dictionary entry together with the name of its **direct hypernym** and its **grandhypernym**.\n\nThe result must contain three columns:\n\n* **entry** — the value of word for the dictionary entry.\n* **hypernym** — the direct hypernym of the entry.\n* **grandhypernym** — the direct hypernym of the entry’s hypernym.\n\nOnly include entries that have **both a direct hypernym and a grandhypernym**.\n',
    validator=val_practice_2,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['dictionary']
)


In [19]:
# @title Practice 3
base_practice_3 = make_df_validator_nospoilers(
    expected_hash='a10e94a5495f25b898353b91a1fe4bb539c9b11d8f9c616a71511b70e9dc4d59',
    required_cols=['entry', 'hypernym', 'grandhypernym'],
    expected_rows=19,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_3 = base_practice_3

make_sql_runner(
    conn,
    runner_id="practice_3",
    description_md='### Practice 3\nWrite a query that lists each dictionary entry together with its **direct hypernym** and its **grandhypernym**.\n\nThe result must contain three columns:\n\n* **entry** — the value of word for the dictionary entry.\n* **hypernym** — the direct hypernym of the entry.\n* **grandhypernym** — the direct hypernym of the entry’s hypernym.\n\nInclude **all dictionary entries**, even if an entry has no hypernym or the hypernym has no hypernym.\n',
    validator=val_practice_3,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['dictionary']
)


## Non primary–foreign key JOINs

Up to this point, we’ve mostly joined tables using **primary key–foreign key relationships**.

In most SQL queries, the `ON` condition connects:
- the **primary key** of one table  
- with the **foreign key** of another table.

However, this is not the only way to join tables.

SQL allows you to join tables using **any columns**, as long as the condition makes sense. The columns don’t have to be keys — they simply need to contain values that can be matched between the tables.

So far, even when joining tables on non primary–foreign key columns, we have still used the **equality operator (`=`)**. But equality is not the only option.

In fact, the `ON` clause can contain **any logical condition**, including comparison operators such as:

- `<`
- `>`
- `<=`
- `>=`
- `!=` or `<>`

You can also use other expressions such as `BETWEEN`, or more complex logical conditions.

---

### Example

Suppose we have two tables: `student` and `subject`.

- The table `subject` contains all subjects offered at a university.
- The table `student` contains information about students and the subjects they are currently studying.

Now imagine we want to show **which subjects each student could switch to**. In other words, we want to display every subject **except the one the student is currently studying**.

We can write the following query:

```sql
SELECT
  student.name,
  subject.name
FROM student
JOIN subject
  ON student.subject_id != subject.id;
````

This query joins **each student with every subject that is different from their current subject**.

As a result:

* every student is paired with all subjects
* **except** the one they are already studying.


## Tables for the next exercises

For the next set of exercises, we’ve prepared two tables: `item` and `buyer`.

---

### `item`

Let’s start with the table `item`.

It contains the following columns:

- `id` – the unique identifier of the item  
- `name` – the name of the item  
- `price` – the price of the item  
- `type` – the type or category of the item

---

### `buyer`

The second table is `buyer`.

It contains the following columns:

- `id` – the unique identifier of the buyer  
- `name` – the name of the buyer  
- `funds` – the amount of money the buyer currently has available


In [20]:
# @title Practice 4
base_practice_4 = make_df_validator_nospoilers(
    expected_hash='ea5a4d5df8ee488d670580f08ec891d5f21c780473534b311fd1d5b50c672f7a',
    required_cols=['name', 'name_1'],
    expected_rows=29,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_4 = base_practice_4

make_sql_runner(
    conn,
    runner_id="practice_4",
    description_md='### Practice 4\nWrite a query that lists every **buyer** together with the **items they can afford**.\n\nThe result must contain two columns:\n\n* **buyer** — the name of the buyer.\n* **item** — the name of the item.\n\nInclude **all buyer–item combinations** where the buyer has enough funds to purchase the item.\n',
    validator=val_practice_4,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['item', 'buyer']
)


In [21]:
# @title Practice 5
base_practice_5 = make_df_validator_nospoilers(
    expected_hash='cf16d2bf45c390436201701271c521c8a393504e7f8551d14cdff196f03cca9d',
    required_cols=['name', 'name_1'],
    expected_rows=17,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_5 = base_practice_5

make_sql_runner(
    conn,
    runner_id="practice_5",
    description_md="### Practice 5\nWrite a query that lists buyers together with the items they can afford.\n\nOnly include buyers whose **funds are greater than 10 000**.\nOnly include items whose **type is not `'painting'`**.\n\nThe result must contain two columns:\n\n* **buyer** — the `name` of the buyer.\n* **item** — the `name` of the item.\n\nInclude all buyer–item combinations that satisfy these conditions.\n",
    validator=val_practice_5,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['item', 'buyer']
)


## Apartments and Couples Dataset

Great! Let’s look at two more tables. This time we’ll work with **apartments** and **couples** who are looking for a place to rent.

---

### `apartments`

This table contains information about available apartments.

It has the following columns:

- `id` – the unique identifier of the apartment  
- `location` – the city where the apartment is located  
- `price` – the price of the apartment  
- `rating` – the rating of the apartment

---

### `couples`

Now let’s look at the table `couples`, which contains information about couples searching for an apartment.

It has five columns:

- `id` – the unique identifier of the couple  
- `couple_name` – the name of the couple (both partners’ names combined)  
- `min_price` – the minimum price of apartments the couple is willing to consider  
- `max_price` – the maximum price the couple is willing to pay  
- `pref_location` – the location where the couple would prefer to rent an apartment


In [22]:
# @title Practice 6
base_practice_6 = make_df_validator_nospoilers(
    expected_hash='08b089ee96aaa38e4959830368429411ff5de7b53c595795caf14abade082310',
    required_cols=['couple_name', 'id'],
    expected_rows=29,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_6 = base_practice_6

make_sql_runner(
    conn,
    runner_id="practice_6",
    description_md='### Practice 6\nWrite a query that lists couples together with the **IDs of apartments within their price range**.\n\nThe result must contain two columns:\n\n* **couple** — the couple_name from the couples table.\n* **apartment_id** — the id of the apartment.\n\nInclude **all couple–apartment combinations** where the apartment’s price is within the couple’s price range.\n',
    validator=val_practice_6,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['couples', 'apartments']
)


In [23]:
# @title Practice 7
base_practice_7 = make_df_validator_nospoilers(
    expected_hash='14c743563fbe99f52e856b99dcd9647739f17aa15d2fbeadeeb0a089a8c2f8e3',
    required_cols=['couple_name', 'id', 'location'],
    expected_rows=8,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_7 = base_practice_7

make_sql_runner(
    conn,
    runner_id="practice_7",
    description_md='### Practice 7\nWrite a query that lists couples together with apartments that **match their requirements**.\n\nAn apartment matches a couple if:\n\n* its **price is within the couple’s price range**, and\n* its **location is the couple’s preferred location**.\n\nThe result must contain three columns:\n\n* **couple** — the couple_name from the couples table.\n* **apartment_id** — the id of the apartment.\n* **location** — the location of the apartment.\n\nInclude **all couple–apartment combinations** that satisfy these conditions.\n',
    validator=val_practice_7,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['apartments', 'couples']
)


In [24]:
# @title Practice 8
base_practice_8 = make_df_validator_nospoilers(
    expected_hash='95f5d9657c7ed1cc8adb8a7803f5c5616a9a043401bc7bf1a99a695dafbcd844',
    required_cols=['couple', 'preferred_location', 'apartment_id'],
    expected_rows=72,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_8 = base_practice_8

make_sql_runner(
    conn,
    runner_id="practice_8",
    description_md='### Practice 8\nWrite a query that lists couples together with apartments that **do not match their requirements**.\n\nAn apartment does **not** match a couple if:\n\n* its **price is outside the couple’s price range**, and\n* its **location is different from the couple’s preferred location**.\n\nThe result must contain three columns:\n\n* **couple** — the couple_name from the couples table.\n* **preferred_location** — the couple’s pref_location.\n* **apartment_id** — the id of the apartment.\n\nInclude **all couple–apartment combinations** that satisfy these conditions.\n',
    validator=val_practice_8,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['apartments', 'couples']
)
